In [2]:
import os
import matplotlib.pyplot as plt
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN

OUT_DIR = "olympiad_summary"
CHART_DIR = os.path.join(OUT_DIR, "charts")
os.makedirs(CHART_DIR, exist_ok=True)

plt.rcParams["font.family"] = "DejaVu Sans"

# ---------- Данные из ноутбука ----------
grades = [5, 6, 7, 8, 9, 10, 11]
grade_scores = [4.50, 4.85, 5.26, 5.46, 5.91, 6.21, 6.62]

regions_sorted = [
    "Пермь", "Краснодар", "Казань", "Уфа", "Воронеж",
    "Санкт-Петербург", "Ростов-на-Дону", "Екатеринбург",
    "Москва", "Новосибирск"
]
region_scores = [5.38, 5.43, 5.48, 5.49, 5.50, 5.58, 5.60, 5.62, 5.66, 5.73]

tasks = ["Task 1", "Task 9", "Task 3", "Task 2", "Task 6", "Task 11",
         "Task 10", "Task 12", "Task 7", "Task 4", "Task 8", "Task 5"]
solve_rate = [29.8, 32.0, 33.9, 37.8, 45.1, 46.7, 53.3, 57.2, 58.7, 59.2, 59.8, 63.9]
avg_time = [64.1, 65.9, 66.0, 69.1, 71.7, 72.8, 75.9, 77.8, 79.1, 78.8, 79.5, 81.2]

# ---------- Графики ----------
# 1. Средний балл по классам
plt.figure(figsize=(8, 4.5))
bars = plt.bar([str(g) for g in grades], grade_scores, color="#4C78A8")
plt.title("Средний балл по классам", fontsize=14)
plt.xlabel("Класс")
plt.ylabel("Средний балл")
plt.ylim(0, 8)
for b, v in zip(bars, grade_scores):
    plt.text(b.get_x() + b.get_width()/2, v + 0.1, f"{v:.2f}", ha="center", fontsize=10)
plt.tight_layout()
grade_chart = os.path.join(CHART_DIR, "grade_avg.png")
plt.savefig(grade_chart, dpi=200)
plt.close()

# 2. Средний балл по регионам
plt.figure(figsize=(8, 5))
bars = plt.barh(regions_sorted, region_scores, color="#F58518")
plt.title("Средний балл по регионам", fontsize=14)
plt.xlabel("Средний балл")
plt.xlim(5.0, 6.0)
for b, v in zip(bars, region_scores):
    plt.text(v + 0.01, b.get_y() + b.get_height()/2, f"{v:.2f}", va="center", fontsize=10)
plt.tight_layout()
region_chart = os.path.join(CHART_DIR, "region_avg.png")
plt.savefig(region_chart, dpi=200)
plt.close()

# 3. Solve rate по заданиям
plt.figure(figsize=(8, 5))
bars = plt.barh(tasks, solve_rate, color="#54A24B")
plt.title("Доля правильных ответов по заданиям, %", fontsize=14)
plt.xlabel("Solve rate, %")
plt.xlim(0, 75)
for b, v in zip(bars, solve_rate):
    plt.text(v + 1, b.get_y() + b.get_height()/2, f"{v:.1f}%", va="center", fontsize=10)
plt.tight_layout()
task_chart = os.path.join(CHART_DIR, "task_solve_rate.png")
plt.savefig(task_chart, dpi=200)
plt.close()

# 4. Время vs solve rate
plt.figure(figsize=(8, 5))
plt.scatter(avg_time, solve_rate, color="#E45756", s=80)
for x, y, t in zip(avg_time, solve_rate, tasks):
    plt.annotate(t.replace("Task ", "T"), (x, y), textcoords="offset points", xytext=(5, 5), fontsize=9)
plt.title("Среднее время и доля правильных ответов", fontsize=14)
plt.xlabel("Среднее время, сек")
plt.ylabel("Solve rate, %")
plt.grid(alpha=0.3)
plt.tight_layout()
scatter_chart = os.path.join(CHART_DIR, "time_vs_solve.png")
plt.savefig(scatter_chart, dpi=200)
plt.close()

# ---------- PPTX ----------
prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)
BLANK = prs.slide_layouts[6]

def add_title(slide, text):
    box = slide.shapes.add_textbox(Inches(0.6), Inches(0.3), Inches(12.1), Inches(0.8))
    tf = box.text_frame
    tf.text = text
    p = tf.paragraphs[0]
    p.font.size = Pt(30)
    p.font.bold = True
    p.font.color.rgb = RGBColor(31, 78, 121)

def add_bullets(slide, items, left=0.8, top=1.4, width=11.8, height=5.6, size=20):
    box = slide.shapes.add_textbox(Inches(left), Inches(top), Inches(width), Inches(height))
    tf = box.text_frame
    tf.word_wrap = True
    for i, item in enumerate(items):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.text = "• " + item
        p.font.size = Pt(size)
        p.space_after = Pt(8)

def add_image_slide(title, image_path, bullets):
    slide = prs.slides.add_slide(BLANK)
    add_title(slide, title)
    slide.shapes.add_picture(image_path, Inches(0.6), Inches(1.45), width=Inches(7.3))
    add_bullets(slide, bullets, left=8.2, top=1.45, width=4.5, height=5.5, size=17)

def add_text_slide(title, bullets):
    slide = prs.slides.add_slide(BLANK)
    add_title(slide, title)
    add_bullets(slide, bullets)

# 1. Титул
slide = prs.slides.add_slide(BLANK)
box = slide.shapes.add_textbox(Inches(0.8), Inches(2.2), Inches(11.7), Inches(1.4))
tf = box.text_frame
tf.text = "Итоги олимпиады: ключевые выводы"
p = tf.paragraphs[0]
p.font.size = Pt(40)
p.font.bold = True
p.font.color.rgb = RGBColor(31, 78, 121)
p.alignment = PP_ALIGN.CENTER

box2 = slide.shapes.add_textbox(Inches(0.8), Inches(3.7), Inches(11.7), Inches(1.0))
tf2 = box2.text_frame
tf2.text = "Анализ 8 000 участников · 10 регионов · 5–11 классы"
p2 = tf2.paragraphs[0]
p2.font.size = Pt(24)
p2.alignment = PP_ALIGN.CENTER

# 2. Данные и очистка
add_text_slide("Данные и очистка", [
    "8 037 строк → 8 000 уникальных участников",
    "Удалено 37 дублей по student_id",
    "Названия регионов нормализованы до 10",
    "Удалено 72 сессии короче 1 минуты — 0,9% данных",
    "Итоговая выборка: 7 928 участников",
    "Проверено: total_score совпадает с суммой правильных ответов"
])

# 3. Сбалансированная выборка
add_text_slide("Сбалансированная выборка", [
    "Классы 5–11: 1 093–1 160 участников на класс",
    "10 регионов: 745–838 участников на регион",
    "590 школ, в среднем 13,4 участника на школу",
    "Медиана по школам — 13 участников",
    "Половина школ — 11–16 участников"
])

# 4. Класс
add_image_slide("Ключевой вывод 1: класс сильнее всего связан с результатом", grade_chart, [
    "Средний балл растёт с 4,50 в 5 классе до 6,62 в 11 классе",
    "Разница — +2,12 балла",
    "Тренд сохраняется во всех регионах",
    "Один тест для 5–11 классов выглядит спорно"
])

# 5. Регионы
add_image_slide("Ключевой вывод 2: регионы различаются слабо", region_chart, [
    "Диапазон средних баллов: 5,38–5,73",
    "Разница — всего 0,35 балла",
    "Устойчивый рейтинг регионов по этим данным строить нельзя",
    "Лучшие значения: Новосибирск, Москва, Екатеринбург",
    "Нижние значения: Пермь, Краснодар, Казань"
])

# 6. Устройства
add_text_slide("Ключевой вывод 3: тип устройства не влияет на результат", [
    "Desktop — 59,4%, mobile — 35,5%, tablet — 5,1%",
    "Средний балл: desktop — 5,53; mobile — 5,57; tablet — 5,60",
    "Число отвеченных заданий практически одинаково: 11,52–11,54",
    "Отключения: около 0,19–0,20 на участника",
    "Вывод: mobile не создаёт disadvantage, поддержку важно сохранять"
])

# 7. Сложность заданий
add_image_slide("Сложность заданий", task_chart, [
    "Самые сложные: Task 1 — 29,8%, Task 9 — 32,0%, Task 3 — 33,9%, Task 2 — 37,8%",
    "Самый простой: Task 5 — 63,9%",
    "Task 6 и Task 11 — средний уровень: 45,1% и 46,7%",
    "Задания 1, 2, 3 и 9 требуют отдельного методического анализа"
])

# 8. Время и сложность
add_image_slide("Время и сложность: очень сильная связь", scatter_chart, [
    "Корреляция solve rate и среднего времени — 0,998",
    "Чем больше времени в среднем тратят участники, тем выше доля правильных ответов",
    "Task 1: 64,1 сек — самый низкий solve rate",
    "Task 5: 81,2 сек — самый высокий solve rate",
    "Связь линейная и очень устойчивая"
])

# 9. Пропуски
add_text_slide("Пропуски заданий почти равномерны", [
    "Skip rate по заданиям: 3,6–4,3%",
    "Нет признаков массового пропуска именно самых сложных заданий",
    "Сложность заданий скорее содержательная, а не следствие избегания",
    "Это усиливает рекомендацию пересмотреть формулировки и сложность Task 1, 2, 3 и 9"
])

# 10. Школы
add_text_slide("Школы: 590, в среднем 13,4 участника", [
    "Медиана — 13 участников",
    "Половина школ — 11–16 участников",
    "Диапазон — 4–29 участников",
    "88 школ имели меньше 10 участников",
    "Для рейтингов школ корректно использовать только школы с ≥10 участниками — 502 школы"
])

# 11. Главные выводы
add_text_slide("Главные выводы", [
    "Выборка сбалансирована по классам, регионам и устройствам",
    "Класс объясняет результаты сильнее, чем регион и устройство",
    "Региональные различия малы: 5,38–5,73",
    "Самые сложные задания: Task 1, 2, 3 и 9",
    "Один тест для 5–11 классов выглядит спорно",
    "Сессии короче 1 минуты — сигнал для проверки качества данных"
])

# 12. Рекомендации
add_text_slide("Рекомендации", [
    "Образовательной команде: пересмотреть задания 1, 2, 3 и 9 — сложность и формулировки",
    "Образовательной команде: рассмотреть отдельные тесты или отдельную интерпретацию результатов по возрастным группам",
    "Продуктовой команде: добавить автоматическую проверку сессий <1 минуты",
    "Продуктовой команде: сохранять поддержку mobile — около трети участников, результаты сопоставимы с desktop",
    "Аналитике: для рейтингов школ использовать порог ≥10 участников"
])

pptx_path = os.path.join(OUT_DIR, "olympiad_summary.pptx")
prs.save(pptx_path)
print(f"Готово: {pptx_path}")
print("Для PDF выполните:")
print(f"libreoffice --headless --convert-to pdf {pptx_path}")

Готово: olympiad_summary\olympiad_summary.pptx
Для PDF выполните:
libreoffice --headless --convert-to pdf olympiad_summary\olympiad_summary.pptx
